### For all the solvation entropy equations we will need three values - the molarity of our solvent, and the VdW volumes of our solvent and solute. The molarity is needed for computing "vfree" (free space per solvent molecule). First let's initialize our `StructureVolume` objects

In [ ]:
from ase.build import molecule
from pymatgen.io.ase import AseAtomsAdaptor
from JDFTxFreeNrg.volume import StructureVolume
from os import getcwd
from pathlib import Path

h2o_cache = Path(getcwd()) / "data" / "H2O_cache"
acetic_acid_cache = Path(getcwd()) / "data" / "CH3COOH_cache"

water_struct = AseAtomsAdaptor.get_structure(molecule("H2O", vacuum=10.0))
acetic_acid_struct = AseAtomsAdaptor.get_structure(molecule("CH3COOH", vacuum=10.0))
water_sv = StructureVolume.from_structure(water_struct, cache_parent=h2o_cache, method="Mesh")
acetic_acid_sv = StructureVolume.from_structure(acetic_acid_struct, cache_parent=acetic_acid_cache, method="Mesh")
water_sv.clear_cache()
acetic_acid_sv.clear_cache()

# Then compute our required values 

In [18]:
from JDFTxFreeNrg.solv_entropy import get_vfree

water_vol = water_sv.get_volume(npoints=1e7)
water_vfree = get_vfree(water_vol, 55.5)
acetic_acid_vol = acetic_acid_sv.get_volume(npoints=1e7)


### `get_solv_entropy_trans` requires the above values, along with a `Structure` for the solute (but also accepts a `StructureVolume`), the temperature (in K), and the dimensionality of the free translations (most likely 3, but you can change to 1 or 2 for partially restricted solutes)

In [19]:
from JDFTxFreeNrg.solv_entropy import get_solv_entropy_trans, J_to_eV

import numpy as np

T = 300.
solv_acetic_acid_entropy_trans = get_solv_entropy_trans(acetic_acid_sv, acetic_acid_vol, water_vol, water_vfree, T, d=3)
print(f"Solvation entropy transfer of acetic acid at {T} K: {solv_acetic_acid_entropy_trans} eV/K")

Solvation entropy transfer of acetic acid at 300.0 K: 0.0012081364394594602 eV/K


### `get_solv_entropy_rot` doesn't require the solvent volume, only the free volume

In [20]:
from JDFTxFreeNrg.solv_entropy import get_solv_entropy_rot

solv_acetic_acid_entropy_rot = get_solv_entropy_rot(acetic_acid_sv, acetic_acid_vol, water_vfree, T)
print(f"Solvation entropy rotational of acetic acid at {T} K: {solv_acetic_acid_entropy_rot} eV/K")

Solvation entropy rotational of acetic acid at 300.0 K: 0.0008942824108689126 eV/K


### The general rule of thumb is that solvated entropy should be ~60% that of gas phase entropy. This package also offers equivalent functions for gas phase entropies in `JDFTxFreeNrg.standard` to be used in the appropriate setting, but is here used for benchmarking ∆S_solv. Note there is not solvated forms for any enthalpies, nor contributions to vibrational free energy

In [21]:
from JDFTxFreeNrg.standard import get_entropy_trans, get_ideal_gas_vol, get_entropy_rot


gas_acetic_acid_entropy_trans = get_entropy_trans(np.sum([site.specie.atomic_mass for site in acetic_acid_sv.sites]), T, get_ideal_gas_vol(1.0, T), d=3)
gas_acetic_acid_entropy_rot = get_entropy_rot(acetic_acid_sv, T)
gas_acetic_acid_entropy = gas_acetic_acid_entropy_trans + gas_acetic_acid_entropy_rot
solv_acetic_acid_entropy = solv_acetic_acid_entropy_trans + solv_acetic_acid_entropy_rot

print(f"{(solv_acetic_acid_entropy / gas_acetic_acid_entropy)*100:.2f}% of the gas phase entropy is retained upon solvation.")

78.06% of the gas phase entropy is retained upon solvation.
